# state_objective_state_conditioned_loss_v1

Purpose: test a state-conditioned BC loss screen before any walk-forward PPO handoff.

Context:
- Previous `state_objective_rebound_weighted_bc_v1` raised global/downtrend fit but kept original rebound accuracy at `0.429`.
- Previous `state_objective_rebound_target_enrichment_v1` expanded rebound target rows to `19`, but original rebound accuracy stayed `0.429`.
- This notebook tests whether static state/action sample weights can improve rebound fit without relying on a neighbor-label grid or a single in-sample metric.

External references used for this mechanism:
- Qlib: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Focal loss / hard-example weighting: https://arxiv.org/abs/1708.02002
- Class-balanced loss by effective sample count: https://arxiv.org/abs/1901.05555
- Demonstration-guided RL / supervised auxiliary signal before RL: https://arxiv.org/abs/1704.03732

Governance:
- Baseline: `state_objective_rebound_target_enrichment_v1` and `state_objective_rebound_weighted_bc_v1`.
- Role: diagnose, not promote.
- Changed layer: model-selection/training objective loss weighting.
- Target: original rebound accuracy >= 0.60 with no global/downtrend collapse, then only allow a future walk-forward PPO handoff.
- Failure mode: in-sample rebound improves but cross-fold holdout or ordinary/downtrend fit degrades, indicating overfit.


In [1]:

from __future__ import annotations

import json
import random
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor,
    fit_bc_actor,
    state_objective_bc_sample_weights,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy

EXPERIMENT = 'state_objective_state_conditioned_loss_v1'
OUT_DIR = Path('runtime/model_selection/card') / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_TARGETS = Path('runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv')
SOURCE_FRAME = Path('data/soxl_profile_balanced_rule_residual_v22.pkl')
BASELINE_ENRICHMENT = Path('runtime/model_selection/card/state_objective_rebound_target_enrichment_v1/selection.json')
BASELINE_WEIGHTED = Path('runtime/model_selection/card/state_objective_rebound_weighted_bc_v1/selection.json')
PROFILE_NAME = 'risk_budget_usage_v1'
RNG_SEED = 23

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

source_targets = pd.read_csv(SOURCE_TARGETS)
source_targets['datetime'] = pd.to_datetime(source_targets['datetime'])
feature_frame = pd.read_pickle(SOURCE_FRAME)
profile = load_profile(PROFILE_NAME)
with BASELINE_ENRICHMENT.open() as fh:
    enrichment_selection = json.load(fh)
with BASELINE_WEIGHTED.open() as fh:
    weighted_selection = json.load(fh)

dataset = build_state_objective_bc_dataset_from_frame(
    feature_frame,
    targets=source_targets,
    profile=profile,
    sample_size=0,
    rng_seed=RNG_SEED,
)
metadata = dataset.metadata.copy()
metadata['fold'] = metadata['fold'].astype(str)
metadata['slice'] = np.select(
    [metadata['is_rebound_cluster'], metadata['is_downtrend_state']],
    ['rebound', 'downtrend'],
    default='ordinary',
)

base_context = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'profile': PROFILE_NAME,
    'source_frame': str(SOURCE_FRAME),
    'source_targets': str(SOURCE_TARGETS),
    'samples': int(len(dataset.actions)),
    'fold_counts': {str(k): int(v) for k, v in metadata['fold'].value_counts().to_dict().items()},
    'slice_counts': {str(k): int(v) for k, v in metadata['slice'].value_counts().to_dict().items()},
    'baseline_enrichment_rebound_accuracy': float(enrichment_selection['best_original_rebound_accuracy']),
    'baseline_weighted_rebound_accuracy': float(weighted_selection['best_rebound_accuracy']),
}
print(json.dumps(base_context, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "baseline_enrichment_rebound_accuracy": 0.42857142857142855,
  "baseline_weighted_rebound_accuracy": 0.42857142857142855,
  "created_at": "2026-05-25T13:00:45.186488+00:00",
  "experiment": "state_objective_state_conditioned_loss_v1",
  "fold_counts": {
    "wf_2022_train": 505,
    "wf_2023_2024_train": 250
  },
  "profile": "risk_budget_usage_v1",
  "samples": 755,
  "slice_counts": {
    "downtrend": 476,
    "ordinary": 275,
    "rebound": 4
  },
  "source_frame": "data/soxl_profile_balanced_rule_residual_v22.pkl",
  "source_targets": "runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv"
}


In [2]:

CONFIGS = [
    ('baseline_unweighted', StateObjectiveBCSampleWeightConfig()),
    ('rebound_x4_cb099', StateObjectiveBCSampleWeightConfig(rebound_multiplier=4.0, class_balanced_beta=0.99)),
    ('rebound_x8_cb099', StateObjectiveBCSampleWeightConfig(rebound_multiplier=8.0, class_balanced_beta=0.99)),
    ('rebound_x12_cb099', StateObjectiveBCSampleWeightConfig(rebound_multiplier=12.0, class_balanced_beta=0.99)),
    ('rebound_x8_downtrend_x1p5_cb099', StateObjectiveBCSampleWeightConfig(rebound_multiplier=8.0, downtrend_multiplier=1.5, class_balanced_beta=0.99)),
    ('rebound_x12_downtrend_x1p5_cb099', StateObjectiveBCSampleWeightConfig(rebound_multiplier=12.0, downtrend_multiplier=1.5, class_balanced_beta=0.99)),
    ('rebound_x8_objpow025_cb099', StateObjectiveBCSampleWeightConfig(rebound_multiplier=8.0, objective_weight_power=0.25, class_balanced_beta=0.99)),
    ('rebound_x12_objpow025_cb099', StateObjectiveBCSampleWeightConfig(rebound_multiplier=12.0, objective_weight_power=0.25, class_balanced_beta=0.99)),
    ('rebound_x8_downtrend_guard_cb099', StateObjectiveBCSampleWeightConfig(rebound_multiplier=8.0, downtrend_multiplier=0.75, class_balanced_beta=0.99)),
]
TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=600, lr=1e-3, batch_size=128, seed=RNG_SEED)
FOLDS = sorted(metadata['fold'].unique().tolist())

print('folds', FOLDS)
print('configs', [name for name, _ in CONFIGS])


folds ['wf_2022_train', 'wf_2023_2024_train']
configs ['baseline_unweighted', 'rebound_x4_cb099', 'rebound_x8_cb099', 'rebound_x12_cb099', 'rebound_x8_downtrend_x1p5_cb099', 'rebound_x12_downtrend_x1p5_cb099', 'rebound_x8_objpow025_cb099', 'rebound_x12_objpow025_cb099', 'rebound_x8_downtrend_guard_cb099']


In [3]:

def make_actor(seed_offset: int = 0):
    torch.manual_seed(RNG_SEED + seed_offset)
    np.random.seed(RNG_SEED + seed_offset)
    random.seed(RNG_SEED + seed_offset)
    _policy, actor = build_policy(observation_dim_for_profile(profile), profile)
    return actor


def eval_slice_rows(actor, scope: str, mask: np.ndarray, config_name: str) -> list[dict[str, object]]:
    rows = []
    scope_meta = metadata.loc[mask].reset_index(drop=True)
    scope_obs = dataset.obs[mask]
    scope_actions = dataset.actions[mask]
    for slice_name, slice_mask in {
        'all': np.ones(len(scope_meta), dtype=bool),
        'rebound': scope_meta['is_rebound_cluster'].to_numpy(dtype=bool),
        'downtrend': scope_meta['is_downtrend_state'].to_numpy(dtype=bool),
        'ordinary': (~scope_meta['is_rebound_cluster'] & ~scope_meta['is_downtrend_state']).to_numpy(dtype=bool),
    }.items():
        metrics = evaluate_bc_actor(actor, scope_obs[slice_mask], scope_actions[slice_mask])
        rows.append({
            'config': config_name,
            'scope': scope,
            'slice': slice_name,
            'samples': int(metrics['samples']),
            'loss': float(metrics['loss']) if pd.notna(metrics['loss']) else np.nan,
            'accuracy': float(metrics['accuracy']) if pd.notna(metrics['accuracy']) else np.nan,
        })
    return rows


def summarize_weights(config_name: str, weights: np.ndarray) -> list[dict[str, object]]:
    rows = []
    frame = metadata.copy()
    frame['weight'] = weights
    for slice_name, group in frame.groupby('slice', dropna=False):
        rows.append({
            'config': config_name,
            'slice': str(slice_name),
            'samples': int(len(group)),
            'weight_mean': float(group['weight'].mean()),
            'weight_min': float(group['weight'].min()),
            'weight_max': float(group['weight'].max()),
        })
    return rows

screen_rows = []
slice_rows = []
weight_rows = []

for config_idx, (config_name, weight_config) in enumerate(CONFIGS):
    weights = state_objective_bc_sample_weights(metadata, actions=dataset.actions, config=weight_config)
    weight_rows.extend(summarize_weights(config_name, weights))

    actor = make_actor(config_idx * 10)
    fit_metrics = fit_bc_actor(
        actor,
        dataset.obs,
        dataset.actions,
        TRAIN_CONFIG,
        sample_weight_array=weights,
    )
    full_mask = np.ones(len(dataset.actions), dtype=bool)
    slice_rows.extend(eval_slice_rows(actor, 'full_fit', full_mask, config_name))

    holdout_summary: dict[str, float] = {}
    fold_rebound_accuracies = []
    fold_all_accuracies = []
    for fold_idx, heldout_fold in enumerate(FOLDS):
        train_mask = (metadata['fold'] != heldout_fold).to_numpy(dtype=bool)
        eval_mask = (metadata['fold'] == heldout_fold).to_numpy(dtype=bool)
        fold_actor = make_actor(config_idx * 10 + fold_idx + 1)
        fit_bc_actor(
            fold_actor,
            dataset.obs[train_mask],
            dataset.actions[train_mask],
            TRAIN_CONFIG,
            sample_weight_array=weights[train_mask],
        )
        fold_scope = f'holdout_{heldout_fold}'
        rows = eval_slice_rows(fold_actor, fold_scope, eval_mask, config_name)
        slice_rows.extend(rows)
        row_by_slice = {row['slice']: row for row in rows}
        fold_rebound_accuracies.append(float(row_by_slice['rebound']['accuracy']))
        fold_all_accuracies.append(float(row_by_slice['all']['accuracy']))
        holdout_summary[f'{heldout_fold}_rebound_accuracy'] = float(row_by_slice['rebound']['accuracy'])
        holdout_summary[f'{heldout_fold}_all_accuracy'] = float(row_by_slice['all']['accuracy'])

    full_eval = {row['slice']: row for row in eval_slice_rows(actor, 'screen_summary', full_mask, config_name)}
    screen_row = {
        'config': config_name,
        **{f'param_{k}': v for k, v in asdict(weight_config).items()},
        **{f'fit_{k}': v for k, v in fit_metrics.items()},
        'full_all_accuracy': float(full_eval['all']['accuracy']),
        'full_rebound_accuracy': float(full_eval['rebound']['accuracy']),
        'full_downtrend_accuracy': float(full_eval['downtrend']['accuracy']),
        'full_ordinary_accuracy': float(full_eval['ordinary']['accuracy']),
        'holdout_rebound_accuracy_mean': float(np.nanmean(fold_rebound_accuracies)),
        'holdout_rebound_accuracy_min': float(np.nanmin(fold_rebound_accuracies)),
        'holdout_all_accuracy_mean': float(np.nanmean(fold_all_accuracies)),
        'holdout_all_accuracy_min': float(np.nanmin(fold_all_accuracies)),
        **holdout_summary,
    }
    screen_rows.append(screen_row)

screen = pd.DataFrame(screen_rows)
slice_metrics = pd.DataFrame(slice_rows)
weight_summary = pd.DataFrame(weight_rows)

screen['rebound_improvement_vs_enrichment'] = screen['full_rebound_accuracy'] - base_context['baseline_enrichment_rebound_accuracy']
screen['holdout_rebound_gap_to_gate'] = 0.60 - screen['holdout_rebound_accuracy_min']
screen['overfit_flag'] = screen['full_rebound_accuracy'] > screen['holdout_rebound_accuracy_min'] + 0.20
screen['gate_pass'] = (
    (screen['full_rebound_accuracy'] >= 0.60)
    & (screen['holdout_rebound_accuracy_min'] >= 0.60)
    & (screen['full_all_accuracy'] >= 0.80)
    & (screen['full_downtrend_accuracy'] >= 0.78)
    & (~screen['overfit_flag'])
)
screen['selection_loss'] = (
    screen['holdout_rebound_gap_to_gate'].clip(lower=0) * 10.0
    + (0.80 - screen['full_all_accuracy']).clip(lower=0) * 2.0
    + (0.78 - screen['full_downtrend_accuracy']).clip(lower=0) * 2.0
    + screen['overfit_flag'].astype(float) * 2.5
)

screen = screen.sort_values(['gate_pass', 'selection_loss', 'full_rebound_accuracy'], ascending=[False, True, False]).reset_index(drop=True)

screen.to_csv(OUT_DIR / 'loss_weight_screen.csv', index=False)
slice_metrics.to_csv(OUT_DIR / 'state_slice_metrics.csv', index=False)
weight_summary.to_csv(OUT_DIR / 'weight_summary.csv', index=False)

print('top configs')
display(screen[['config','gate_pass','selection_loss','full_rebound_accuracy','holdout_rebound_accuracy_min','holdout_rebound_accuracy_mean','full_all_accuracy','full_downtrend_accuracy','full_ordinary_accuracy','overfit_flag']])
print('weight summary')
display(weight_summary.head(20))


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


top configs


,config,gate_pass,selection_loss,full_rebound_accuracy,holdout_rebound_accuracy_min,holdout_rebound_accuracy_mean,full_all_accuracy,full_downtrend_accuracy,full_ordinary_accuracy,overfit_flag
0,rebound_x4_cb099,False,8.500000,1.0,0.0,0.0,0.829139,0.870021,0.756364,True
1,rebound_x8_cb099,False,8.500000,1.0,0.0,0.0,0.821192,0.859539,0.752727,True
2,rebound_x12_cb099,False,8.500000,1.0,0.0,0.0,0.833113,0.874214,0.760000,True
3,rebound_x8_downtrend_x1p5_cb099,False,8.500000,1.0,0.0,0.0,0.825166,0.861635,0.760000,True
4,rebound_x12_downtrend_x1p5_cb099,False,8.500000,1.0,0.0,0.0,0.805298,0.867925,0.694545,True
5,rebound_x8_downtrend_guard_cb099,False,8.500000,1.0,0.0,0.0,0.827815,0.870021,0.752727,True
6,baseline_unweighted,False,8.513245,1.0,0.0,0.0,0.793377,0.798742,0.781818,True
7,rebound_x12_objpow025_cb099,False,8.523841,1.0,0.0,0.0,0.788079,0.807128,0.752727,True
8,rebound_x8_objpow025_cb099,False,8.534437,1.0,0.0,0.0,0.782781,0.788260,0.770909,True


weight summary


,config,slice,samples,weight_mean,weight_min,weight_max
0,baseline_unweighted,downtrend,476,1.000000,1.000000,1.000000
1,baseline_unweighted,ordinary,275,1.000000,1.000000,1.000000
2,baseline_unweighted,rebound,4,1.000000,1.000000,1.000000
3,rebound_x4_cb099,downtrend,476,0.797606,0.753914,1.068604
4,rebound_x4_cb099,ordinary,275,1.252655,0.753914,18.231615
5,rebound_x4_cb099,rebound,4,7.714883,4.274417,8.861705
6,rebound_x8_cb099,downtrend,476,0.766285,0.724309,1.026642
7,rebound_x8_cb099,ordinary,275,1.203465,0.724309,17.515688
8,rebound_x8_cb099,rebound,4,14.823863,8.213136,17.027439
9,rebound_x12_cb099,downtrend,476,0.737331,0.696941,0.987851


In [4]:

best = screen.iloc[0].to_dict()
promotion_allowed = bool(best['gate_pass'])
partial_improvement = bool(best['full_rebound_accuracy'] > base_context['baseline_enrichment_rebound_accuracy'])
full_gate = bool(promotion_allowed)

if promotion_allowed:
    decision = 'state_conditioned_loss_gate_pass_requires_walkforward_ppo_no_runtime_promotion'
    next_mechanism = 'walkforward_ppo_with_state_conditioned_bc_loss'
    blocker = 'BC loss screen is not a backtest; it only permits a future walk-forward PPO handoff.'
elif partial_improvement:
    decision = 'state_conditioned_loss_partial_rebound_fit_no_promotion'
    next_mechanism = 'convert_best_loss_to_fold_local_ppo_recipe_then_validate_walkforward'
    blocker = 'In-sample rebound fit improved, but full anti-overfit gate did not pass.'
else:
    decision = 'state_conditioned_loss_no_rebound_improvement_no_promotion'
    next_mechanism = 'explicit_rebound_state_head_or_feature_representation_before_ppo'
    blocker = 'Static state/action sample weights did not improve the original rebound slice enough.'

selection = {
    'experiment': EXPERIMENT,
    'created_at': base_context['created_at'],
    'profile': PROFILE_NAME,
    'source_experiment': 'state_objective_rebound_target_enrichment_v1',
    'direct_comparison_baseline': 'state_objective_rebound_weighted_bc_v1',
    'decision': decision,
    'promotion_allowed': False,
    'full_gate': full_gate,
    'partial_improvement': partial_improvement,
    'promotion_blocker': blocker,
    'best_config': str(best['config']),
    'best_full_rebound_accuracy': float(best['full_rebound_accuracy']),
    'best_holdout_rebound_accuracy_min': float(best['holdout_rebound_accuracy_min']),
    'best_holdout_rebound_accuracy_mean': float(best['holdout_rebound_accuracy_mean']),
    'best_full_all_accuracy': float(best['full_all_accuracy']),
    'best_full_downtrend_accuracy': float(best['full_downtrend_accuracy']),
    'best_full_ordinary_accuracy': float(best['full_ordinary_accuracy']),
    'baseline_rebound_accuracy': float(base_context['baseline_enrichment_rebound_accuracy']),
    'rebound_improvement': float(best['full_rebound_accuracy'] - base_context['baseline_enrichment_rebound_accuracy']),
    'overfit_flag': bool(best['overfit_flag']),
    'next_mechanism': next_mechanism,
    'acceptance_gate_for_next_run': {
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
        'fixed_split_required': True,
        'walkforward_required': True,
        'original_rebound_accuracy_min': 0.60,
        'holdout_rebound_accuracy_min': 0.60,
        'avoid_more_neighbor_enrichment_grids': True,
        'no_runtime_promotion_from_bc_only': True,
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/1708.02002',
        'https://arxiv.org/abs/1901.05555',
        'https://arxiv.org/abs/1704.03732',
    ],
    'sidecars': [
        'base_context.json',
        'loss_weight_screen.csv',
        'state_slice_metrics.csv',
        'weight_summary.csv',
        'summary.json',
    ],
}

summary = {
    'base_context': base_context,
    'best_row': best,
    'selection': selection,
    'conclusion': [
        f"Best config {selection['best_config']} reached full rebound accuracy {selection['best_full_rebound_accuracy']:.3f} vs baseline {selection['baseline_rebound_accuracy']:.3f}.",
        f"Cross-fold holdout rebound min/mean were {selection['best_holdout_rebound_accuracy_min']:.3f}/{selection['best_holdout_rebound_accuracy_mean']:.3f}; anti-overfit gate pass={selection['full_gate']}.",
        f"Full all/downtrend/ordinary accuracies were {selection['best_full_all_accuracy']:.3f}/{selection['best_full_downtrend_accuracy']:.3f}/{selection['best_full_ordinary_accuracy']:.3f}.",
        f"Decision: {selection['decision']}; next mechanism: {selection['next_mechanism']}.",
    ],
}

(OUT_DIR / 'base_context.json').write_text(json.dumps(base_context, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + '\n')

print(json.dumps(selection, indent=2, sort_keys=True))


{
  "acceptance_gate_for_next_run": {
    "avoid_more_neighbor_enrichment_grids": true,
    "fixed_split_required": true,
    "holdout_rebound_accuracy_min": 0.6,
    "max_dd_max_pct": 15.0,
    "no_runtime_promotion_from_bc_only": true,
    "original_rebound_accuracy_min": 0.6,
    "relative_return_min_pp": 30.0,
    "walkforward_required": true
  },
  "baseline_rebound_accuracy": 0.42857142857142855,
  "best_config": "rebound_x4_cb099",
  "best_full_all_accuracy": 0.8291390728476821,
  "best_full_downtrend_accuracy": 0.870020964360587,
  "best_full_ordinary_accuracy": 0.7563636363636363,
  "best_full_rebound_accuracy": 1.0,
  "best_holdout_rebound_accuracy_mean": 0.0,
  "best_holdout_rebound_accuracy_min": 0.0,
  "created_at": "2026-05-25T13:00:45.186488+00:00",
  "decision": "state_conditioned_loss_partial_rebound_fit_no_promotion",
  "direct_comparison_baseline": "state_objective_rebound_weighted_bc_v1",
  "experiment": "state_objective_state_conditioned_loss_v1",
  "external_sourc

## Conclusion

See `selection.json` and `summary.json` generated by the executed cells. This notebook is the primary human-readable record for this single state-conditioned loss screen; sidecars exist for indexing and follow-up automation.
